# Module 29 — Pipeline hand-offs, and lost-in-translation

**THE ONE IDEA:** the **#1 production failure of multi-agent systems** is not reasoning.
It is **formatting**. Agent A's free-text output becomes Agent B's input, A's phrasing
drifts one day, B silently misparses, and the final answer is wrong **but plausible**.

The fix is the same discipline as service-to-service APIs: **a typed contract at every
hand-off.**

Both versions are built here — typed and free-text — and the free-text one is made to
fail on a phrasing change that a human would not even notice.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, re
from pydantic import BaseModel, ValidationError
from _providers import get_client
from _tools import run_tool

client, MODEL, _ = get_client("openai")

class Extracted(BaseModel):          # THE CONTRACT between agent A and agent B
    loan_amount: float
    property_value: float
    year: int

APPLICATION = ("Client wants 285000 against a property valued at 320000. "
               "She is considering repaying in year 2.")

## The typed pipeline — A → B → C

In [ ]:
def agent_a_typed(text) -> Extracted:
    s = Extracted.model_json_schema(); s["additionalProperties"] = False
    r = client.chat.completions.create(model=MODEL, max_tokens=200,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "x", "strict": True, "schema": s}},
        messages=[{"role": "user", "content": "Extract the figures.\n\n" + text}])
    return Extracted.model_validate_json(r.choices[0].message.content)

def agent_b(e: Extracted):           # consumes TYPED fields — no parsing at all
    erc = run_tool("search_policy", {"query": "erc"})
    rate = {1: .05, 2: .04, 3: .03, 4: .02, 5: .01}.get(e.year, 0)
    charge = run_tool("calculate", {"expression": f"{e.loan_amount} * {rate}"})
    return {"ltv": e.loan_amount / e.property_value * 100, "erc": float(charge), "src": erc}

def agent_c(f):
    return (f"LTV {f['ltv']:.1f}%, year-{2} ERC GBP {f['erc']:,.0f}. "
            f"Policy: {f['src'][:44]}")

ext = agent_a_typed(APPLICATION)
print("A ->", ext.model_dump())
facts = agent_b(ext); print("B ->", {k: v for k, v in facts.items() if k != 'src'})
print("C ->", agent_c(facts))

## The free-text pipeline — and the day the phrasing changes

Agent B parses prose with a regex. It works. Then someone tweaks A's prompt, A starts
writing `GBP 285,000` instead of `285000`, and B keeps running.

In [ ]:
def agent_b_freetext(prose):
    nums = re.findall(r"\b(\d{4,})\b", prose.replace(",", "X"))   # comma breaks it
    if len(nums) < 2:
        return None
    loan, prop = float(nums[0]), float(nums[1])
    return {"ltv": loan / prop * 100, "erc": loan * .04}

V1 = "Loan 285000 against property 320000, repaying in year 2."
V2 = "Loan GBP 285,000 against property GBP 320,000, repaying in year 2."

for tag, prose in [("v1 (today)", V1), ("v2 (after a prompt tweak)", V2)]:
    out = agent_b_freetext(prose)
    print(f"  {tag:26} -> {out}")
print("\n^ v2 did not raise. It returned None, or would return a NONSENSE ltv if the")
print("  regex had matched partial digits. Downstream agents act on it regardless.")

## What the contract catches

In [ ]:
broken = '{"loan_amount": "285,000", "property_value": 320000, "year": 2}'
try:
    Extracted.model_validate_json(broken)
except ValidationError as e:
    print("typed hand-off REJECTS it at the boundary:")
    print("  ", e.errors()[0]["loc"], e.errors()[0]["msg"])

print("""
LESSON - this is the #1 silent failure in multi-agent systems, and it has nothing
to do with the models being bad at reasoning.

  FREE TEXT   Agent A writes prose. Agent B parses it. Both are 'working'. Then
              A's phrasing drifts - a prompt tweak, a model upgrade, a different
              temperature - and B misparses. No exception. No log line. The final
              answer is WRONG BUT PLAUSIBLE, which is the worst possible failure.

  TYPED       the schema is validated at the boundary. A drift in phrasing cannot
              change loan_amount from a float into a string without the hand-off
              REFUSING it, loudly, at the exact place it broke.

Treat every agent boundary exactly like a service boundary in microservices:
  - a versioned schema (Pydantic, a tool-call signature, JSON mode)
  - validation ON RECEIPT, not on send
  - a loud failure, never a silent coercion

Free-text inter-agent communication looks elegant in a diagram and is the single
most common cause of multi-agent systems that 'mostly work'.

Module 30 adds the supervisor that ROUTES between agents - and shows what all
this coordination costs.""")

---

**Next:** `30_multi_agent_supervisor.ipynb`